# DumpWatch 2.0 Task B · Final held-out evaluation
The locked test inventory contains only the approved pairs at locations 04 and 10. This notebook reproduces the final held-out scoring for reporting or audit. It reads the frozen scaler and coefficients and performs no model fitting. The decision interval remains -0.5 to +0.5. The zero-threshold forced diagnostic is separate.

Use the files from the final results package as listed in README. Do not reassess labels after viewing scores.

In [ ]:
from pathlib import Path
import os, sys
start = Path(os.environ.get("DUMPWATCH_PACKAGE", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "project_setup.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter within the unpacked code folder or set DUMPWATCH_PACKAGE to it. See SETUP.md.")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from project_setup import project_paths, new_output, enable_demo_vendor
P = project_paths()
INPUTS = ROOT / "Reproduction/Task_B"
REAL_ZIP = P["real_zip"]
SPLIT_CSV = INPUTS / "real_location_split.csv"
FROZEN_DEV_CSV = INPUTS / "frozen_train_development_pair_inventory.csv"
MODEL_JSON = INPUTS / "frozen_real_only_svm.json"
LOCK_JSON = INPUTS / "frozen_test_inventory_lock.json"
TEST_CSV = INPUTS / "frozen_test_pair_inventory.csv"
needed = (REAL_ZIP, SPLIT_CSV, FROZEN_DEV_CSV, MODEL_JSON, LOCK_JSON, TEST_CSV)
missing = [str(p) for p in needed if not p.is_file()]
if missing: raise FileNotFoundError("Missing inputs: " + ", ".join(missing))
OUTPUT_ROOT = new_output("task_b_evaluation")
print("Frozen inference only. Outputs:", OUTPUT_ROOT)
EXPECTED_MODEL_SHA256 = "b6bc6391b3af0e7ea2999e0702876c1afa980f51f83470fcce4238b430f0b6b5"
EXPECTED_TEST_CSV_SHA256 = "c44ce60a76f51bbcb19873499b27ee3cb4a3adc96326b2455dc87d77edbae004"
EXPECTED_DEV_CSV_SHA256 = "bdbb8bca0ce9d8daf398f2ac613090ac2a39be570d05f620733e5ff40c283469"
PREPROCESSING_SHA256 = "643a6534e60d5996c91da1ff5de0235450b08d7631db287011da745a71cdc30d"


## Frozen feature extraction and linear inference
These functions are copied from the frozen preparation notebook. Ground-region crops are set in the locked CSV.

In [ ]:
import csv, hashlib, io, json, zipfile
from collections import Counter, defaultdict
import numpy as np
from PIL import Image

TRAIN = {f"location_{n:02}" for n in (1,3,5,6,7,9)}
DEV = {"location_02", "location_08"}
TEST = {"location_04", "location_10"}
REAL_HASH = "1e97cfe51b4ef7fc9e61aaa7cbd2cbce4ac5a24fd274f07503d7deb5138f0412"
SPLIT_HASH = "0ae3ff815747ba5c8c677fb3670c373AECBE9ECF2B4E3BD6B2410450748A3E60".lower()

def sha(data):
    return hashlib.sha256(data).hexdigest()

def require(ok, message):
    if not ok:
        raise ValueError(message)

def box_from_string(value):
    return [float(n) for n in value.split()]

def descriptor(raw, box):
    left,top,right,bottom=box
    require(0<=left<right<=1 and 0<=top<bottom<=1, f"Invalid normalized ROI {box}")
    with Image.open(io.BytesIO(raw)) as source:
        w,h=source.size
        pixels=source.convert("RGB").crop((round(left*w),round(top*h),round(right*w),round(bottom*h)))
    require(min(pixels.size)>=64, "ROI too small to assess")
    rgb=np.asarray(pixels.resize((128,128),Image.Resampling.BILINEAR),dtype=np.float32)/255
    gray=rgb @ np.array([.299,.587,.114],dtype=np.float32)
    dy,dx=np.gradient(gray)
    magnitude=np.hypot(dx,dy)
    orientation=np.floor((np.arctan2(dy,dx)%np.pi)*9/np.pi).astype(int).clip(0,8)
    features=[]
    for y in range(0,128,32):
        for x in range(0,128,32):
            region=np.s_[y:y+32,x:x+32]
            hist=np.bincount(orientation[region].ravel(),weights=magnitude[region].ravel(),minlength=9)
            features.extend(hist/(np.linalg.norm(hist)+1e-6))
            patch=rgb[region]
            features.extend(patch.mean(axis=(0,1)))
            features.extend(patch.std(axis=(0,1)))
    return np.asarray(features,dtype=np.float32)

def paired_vector(w,a,scores):
    return np.r_[w,a,np.abs(w-a),scores[0],scores[1],scores[0]-scores[1]]

def select_clearance(score, visually_assessable, low, high):
    require(np.isfinite(low) and np.isfinite(high) and low<high,"Invalid abstention interval")
    return np.where(~np.asarray(visually_assessable,dtype=bool),"abstain",
           np.where(score>=high,"fully_cleared",np.where(score<=low,"waste_still_present","abstain")))

def evaluate(target,prediction):
    target,prediction=np.asarray(target),np.asarray(prediction)
    remaining=target==0
    cleared=target==1
    false=int((remaining & (prediction=="fully_cleared")).sum())
    valid=int((prediction!="abstain").sum())
    return dict(false_clearance=false,remaining_total=int(remaining.sum()),
        false_clearance_rate=false/remaining.sum() if remaining.any() else None,
        correctly_cleared=int((cleared & (prediction=="fully_cleared")).sum()),
        cleared_total=int(cleared.sum()),cleared_as_waste_present=int((cleared & (prediction=="waste_still_present")).sum()),
        waste_present_correct=int((remaining & (prediction=="waste_still_present")).sum()),
        evaluated=len(target),coverage=valid/len(target) if len(target) else None,
        abstentions=int(len(target)-valid))

def frozen_score(values,spec):
    values=np.array(values,copy=True)
    # Match StandardScaler's in-place float32 rounding for appearance inputs.
    values-=np.asarray(spec["mean"])
    values/=np.asarray(spec["scale"])
    return values @ np.asarray(spec["coefficients"])+spec["intercept"]

def frozen_pair_features(w,a,model):
    scores=frozen_score(np.stack((w,a)),model["appearance"])
    return paired_vector(w,a,scores)



In [ ]:
import csv,io,json,zipfile
from pathlib import Path
import numpy as np
from PIL import Image

require(sha(REAL_ZIP.read_bytes())==REAL_HASH,"Real ZIP hash mismatch")
require(sha(SPLIT_CSV.read_bytes())==SPLIT_HASH,"Frozen split hash mismatch")
require(sha(FROZEN_DEV_CSV.read_bytes())==EXPECTED_DEV_CSV_SHA256,"Frozen training/development pair CSV changed")
lock=json.loads(LOCK_JSON.read_text(encoding="utf-8"))
require(lock["archive_sha256"]==REAL_HASH and lock["test_pair_inventory_sha256"]==EXPECTED_TEST_CSV_SHA256,
        "Test lock does not match pinned inventory")
test_raw=TEST_CSV.read_bytes()
require(sha(test_raw)==EXPECTED_TEST_CSV_SHA256,"Test pair inventory differs from locked pre-inference CSV")
model_raw=MODEL_JSON.read_bytes()
require(sha(model_raw)==EXPECTED_MODEL_SHA256,"Frozen model bytes differ")
model=json.loads(model_raw)
require(model["version"]=="task-b-frozen-real-only-v1" and model["archive_sha256"]==REAL_HASH
        and model["split_sha256"]==SPLIT_HASH and model["pair_config_sha256"]==EXPECTED_DEV_CSV_SHA256,
        "Frozen model provenance mismatch")
require(model["policy"]=={"low":-.5,"high":.5,"unassessable":"abstain","forced_diagnostic_threshold":0},
        "Frozen model decision policy differs")
test=list(csv.DictReader(io.StringIO(test_raw.decode("utf-8-sig"))))
require(len(test)==len({r["pair_id"] for r in test})==9 and {r["location_id"] for r in test}==TEST
        and len({r["inspection_sha256"] for r in test})==9 and len({r["inspection_decoded_rgb_sha256"] for r in test})==9,
        "Frozen pair uniqueness or membership mismatch")
require(Counter(r["ground_truth"] for r in test)=={"fully_cleared":3,"waste_still_present":6},
        "Frozen class counts differ")
verified=[]
with zipfile.ZipFile(REAL_ZIP) as archive:
    archived={r["relative_path"]:r for r in csv.DictReader(io.StringIO(archive.read("dataset_manifest.csv").decode("utf-8-sig")))}
    for row in test:
        loc,case,stage=row["location_id"],row["case_id"],row["inspection_stage"]
        expected=("fully_cleared" if stage=="A_removed" else "waste_still_present")
        require(row["ground_truth"]==expected and stage in {"A_removed","A_partial"}
                and row["assessment_status"]=="usable" and row["W_sha256"]!=row["inspection_sha256"],
                f"Ground truth or pair role invalid: {row['pair_id']}")
        vectors=[]
        for label,key,h,pixel_key,roi in (
            ("W","W_path","W_sha256","W_decoded_rgb_sha256","W_roi_xyxy"),
            (stage,"inspection_path","inspection_sha256","inspection_decoded_rgb_sha256","inspection_roi_xyxy")):
            path=row[key]
            meta=archived[path]
            require(path.startswith(f"real/{loc}/{case}/") and loc in TEST and meta["location_id"]==loc
                    and meta["case_id"]==case and meta["semantic_stage"]==label,
                    f"Case member mismatch: {path}")
            raw=archive.read(path)
            require(sha(raw)==row[h]==meta["file_hash"],f"Image hash mismatch: {path}")
            with Image.open(io.BytesIO(raw)) as im:
                rgb=im.convert("RGB")
                rgb_hash=sha(f"{rgb.width}x{rgb.height}:".encode()+rgb.tobytes())
            require(rgb_hash==row[pixel_key],f"Decoded image hash mismatch: {path}")
            vectors.append(descriptor(raw,box_from_string(row[roi])))
        verified.append(frozen_pair_features(*vectors,model))
scores=frozen_score(np.stack(verified),model["pair"])
require(scores.shape==(9,) and np.isfinite(scores).all(),"Invalid frozen inference scores")
fixed=select_clearance(scores,np.ones(9,bool),model["policy"]["low"],model["policy"]["high"])
forced=np.where(scores>=0,"fully_cleared","waste_still_present")
predictions=[dict(row,decision_score=float(score),forced_prediction=str(f),fixed_prediction=str(a),
                 abstention_reason="inside fixed (-0.5, +0.5) interval" if a=="abstain" else "")
             for row,score,f,a in zip(test,scores,forced,fixed)]


In [ ]:
from datetime import datetime,timezone
from PIL import ImageDraw,ImageOps
classes=["waste_still_present","fully_cleared"]
pred_labels=classes+["abstain"]
def divide(x,y):return x/y if y else None
def metrics(rows,column):
    matrix=[[sum(r["ground_truth"]==a and r[column]==b for r in rows) for b in pred_labels] for a in classes]
    tn,fp,u0=matrix[0]
    fn,tp,u1=matrix[1]
    accepted=tn+fp+fn+tp
    by={}
    for i,label in enumerate(classes):
        hit=matrix[i][i]
        wrong_prediction=sum(matrix[j][i] for j in range(2) if j!=i)
        wrong_truth=sum(matrix[i][j] for j in range(2) if j!=i)
        by[label]=dict(support=sum(matrix[i]),accepted_support=hit+wrong_truth,
            precision=divide(hit,hit+wrong_prediction),recall_accepted=divide(hit,hit+wrong_truth),
            recall_all=divide(hit,sum(matrix[i])),f1_accepted=divide(2*hit,2*hit+wrong_prediction+wrong_truth),
            correct=hit)
    return dict(n=len(rows),row_order=classes,column_order=pred_labels,confusion_matrix=matrix,
       accepted=accepted,abstained=u0+u1,coverage=divide(accepted,len(rows)),
       accepted_accuracy=divide(tn+tp,accepted),correct_accepted=tn+tp,per_class=by,
       false_clearance_count=fp,false_clearance_denominator=tn+fp+u0,false_clearance_rate=divide(fp,tn+fp+u0),
       false_alarm_count=fn,false_alarm_denominator=fn+tp+u1,false_alarm_rate=divide(fn,fn+tp+u1))

output=OUTPUT_ROOT/datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output.mkdir(parents=True,exist_ok=False)
result=dict(model_sha256=EXPECTED_MODEL_SHA256,test_inventory_sha256=EXPECTED_TEST_CSV_SHA256,
            real_archive_sha256=REAL_HASH,frozen_preparation_csv_sha256=EXPECTED_DEV_CSV_SHA256,
            frozen_notebook_preprocessing_sha256=PREPROCESSING_SHA256,
            development_reference=dict(n=8,forced_correct=6,forced_false_clearance=[1,6],
               forced_false_alarm=[1,2],fixed_correct_accepted=1,fixed_accepted=1,fixed_abstentions=7),
            forced=metrics(predictions,"forced_prediction"),fixed=metrics(predictions,"fixed_prediction"),
            per_location={loc:{name:metrics([r for r in predictions if r["location_id"]==loc],key)
                          for name,key in [("forced","forced_prediction"),("fixed","fixed_prediction")]}
                          for loc in sorted(TEST)})
(output/"final_test_metrics.json").write_text(json.dumps(result,indent=2,allow_nan=False)+"\n",encoding="utf-8")
with (output/"final_test_predictions.csv").open("x",newline="",encoding="utf-8") as f:
    writer=csv.DictWriter(f,fieldnames=list(predictions[0]))
    writer.writeheader();writer.writerows(predictions)

chart=Image.new("RGB",(1080,422),"#f1f5fa")
d=ImageDraw.Draw(chart)
def text(x,y,label,size=19,fill="#172938"):
    d.text((x,y),str(label),fill=fill,font_size=size)
text(22,14,"DumpWatch Task B | Held-out real test confusion matrices",26)
for i,(key,label) in enumerate((("forced","Forced diagnostic  |  zero threshold"),("fixed","Accepted policy  |  fixed +/-0.5"))):
    x=18+i*530
    d.rounded_rectangle((x,62,x+513,401),radius=14,fill="white")
    text(x+16,82,label,21)
    for j,col in enumerate(("Present","Cleared","Abstain")):
        text(x+162+109*j,132,col,16)
    for r,line in enumerate(result[key]["confusion_matrix"]):
        text(x+18,179+r*69,("Waste present","Fully cleared")[r],17)
        for j,n in enumerate(line):
            l,t=x+154+j*109,158+r*69
            fill="#d8eee7" if r==j else "#f1dcd6" if n and j!=2 else "#edf1f7"
            d.rounded_rectangle((l,t,l+97,t+56),radius=6,fill=fill)
            text(l+40,t+14,n,22)
    v=result[key]
    text(x+17,319,f"Accepted {v['accepted']}/9   |   Deferred {v['abstained']}/9",18)
    text(x+17,352,f"False clearance {v['false_clearance_count']}/6   |   False alarms {v['false_alarm_count']}/3",17)
chart.save(output/"final_test_confusion.png")

gallery=Image.new("RGB",(1110,196*len(predictions)+54),"#f1f5fa")
draw=ImageDraw.Draw(gallery)
draw.text((17,10),"Frozen test pairs | W left, inspection right",fill="#172938",font_size=25)
with zipfile.ZipFile(REAL_ZIP) as archive:
    for i,row in enumerate(predictions):
        y=54+196*i
        draw.rounded_rectangle((10,y,1098,y+184),radius=10,fill="white")
        fields=[row["pair_id"],"Truth: "+row["ground_truth"].replace("_"," "),
                "Forced: "+row["forced_prediction"].replace("_"," "),
                f"Fixed: {row['fixed_prediction'].replace('_',' ')} ({row['decision_score']:+.2f})"]
        for j,label in enumerate(fields):
            draw.text((20,y+12+j*34),label,fill="#172938",font_size=16)
        for j,(k,roi) in enumerate((("W_path","W_roi_xyxy"),("inspection_path","inspection_roi_xyxy"))):
            with Image.open(io.BytesIO(archive.read(row[k]))) as im:
                w,h=im.size
                l,t,r,b=box_from_string(row[roi])
                crop=im.convert("RGB").crop((round(l*w),round(t*h),round(r*w),round(b*h)))
            gallery.paste(ImageOps.contain(crop,(355,157)),(355+370*j,y+10))
gallery.save(output/"final_test_prediction_gallery.jpg",quality=95)
print(json.dumps(result,indent=2,allow_nan=False))
print("Held-out outputs:",output)
chart


Reported accuracy with abstention is conditional on accepted decisions. Report coverage beside it. Class recall is also provided over all eligible rows. False clearance and false alarm use the entire corresponding true class as denominator.